In [53]:
import pandas as pd
import re
import numpy as np

In [54]:
# -----------------------------
# 1. Import Project Gutenberg official catalog
# -----------------------------

url = "https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv.gz"

pg = pd.read_csv(url)

# Check original columns
print(pg.columns.tolist())
print(pg.shape)
pg.head()

['Text#', 'Type', 'Issued', 'Title', 'Language', 'Authors', 'Subjects', 'LoCC', 'Bookshelves']
(78999, 9)


,Text#,Type,Issued,Title,Language,Authors,Subjects,LoCC,Bookshelves
0,1,Text,1971-12-01,The Declaration of Independence of the United ...,en,"Jefferson, Thomas, 1743-1826","United States -- History -- Revolution, 1775-1...",E201; JK,Politics; American Revolutionary War; United S...
1,2,Text,1972-12-01,The United States Bill of Rights\r\nThe Ten Or...,en,United States,Civil rights -- United States -- Sources; Unit...,JK; KF,Politics; American Revolutionary War; United S...
2,3,Text,1973-11-01,John F. Kennedy's Inaugural Address,en,"Kennedy, John F. (John Fitzgerald), 1917-1963",United States -- Foreign relations -- 1961-196...,E838,"Category: Essays, Letters & Speeches; Category..."
3,4,Text,1973-11-01,Lincoln's Gettysburg Address\r\nGiven November...,en,"Lincoln, Abraham, 1809-1865",Consecration of cemeteries -- Pennsylvania -- ...,E456,"US Civil War; Category: Essays, Letters & Spee..."
4,5,Text,1975-12-01,The United States Constitution,en,United States,United States -- Politics and government -- 17...,JK; KF,United States; Politics; American Revolutionar...


In [55]:
# -----------------------------
# 2. Clean column names
# -----------------------------

pg = pg.rename(columns={
    "Text#": "gutenberg_id",
    "Type": "type",
    "Issued": "gutenberg_issued",
    "Title": "title",
    "Language": "language",
    "Authors": "author",
    "Subjects": "subjects",
    "LoCC": "locc",
    "Bookshelves": "bookshelves",
})

# Keep only English texts
pg_en = pg[
    pg["language"].astype(str).str.contains(r"\ben\b", na=False)
].copy()

# If Type column exists, keep only text records
if "type" in pg_en.columns:
    pg_en = pg_en[
        pg_en["type"].astype(str).str.lower().eq("text")
    ].copy()

# Basic cleanup
pg_en["subjects"] = pg_en["subjects"].fillna("").astype(str)
pg_en["bookshelves"] = pg_en["bookshelves"].fillna("").astype(str)
pg_en["author"] = pg_en["author"].fillna("").astype(str)
pg_en["title"] = pg_en["title"].fillna("").astype(str)

print(pg_en.shape)
pg_en.head()

(61583, 9)


,gutenberg_id,type,gutenberg_issued,title,language,author,subjects,locc,bookshelves
0,1,Text,1971-12-01,The Declaration of Independence of the United ...,en,"Jefferson, Thomas, 1743-1826","United States -- History -- Revolution, 1775-1...",E201; JK,Politics; American Revolutionary War; United S...
1,2,Text,1972-12-01,The United States Bill of Rights\r\nThe Ten Or...,en,United States,Civil rights -- United States -- Sources; Unit...,JK; KF,Politics; American Revolutionary War; United S...
2,3,Text,1973-11-01,John F. Kennedy's Inaugural Address,en,"Kennedy, John F. (John Fitzgerald), 1917-1963",United States -- Foreign relations -- 1961-196...,E838,"Category: Essays, Letters & Speeches; Category..."
3,4,Text,1973-11-01,Lincoln's Gettysburg Address\r\nGiven November...,en,"Lincoln, Abraham, 1809-1865",Consecration of cemeteries -- Pennsylvania -- ...,E456,"US Civil War; Category: Essays, Letters & Spee..."
4,5,Text,1975-12-01,The United States Constitution,en,United States,United States -- Politics and government -- 17...,JK; KF,United States; Politics; American Revolutionar...


In [113]:
added_canonical_id = [599, 730, 786, 883, 1023, 1400, 6053, 6593, 9182, 9296]
pg_en[pg_en['gutenberg_id'].isin(added_canonical_id)]

,gutenberg_id,type,gutenberg_issued,title,language,author,subjects,locc,bookshelves
580,599,Text,1996-07-01,Vanity Fair,en,"Thackeray, William Makepeace, 1811-1863",Satire; England -- Fiction; Married women -- F...,PR,Historical Fiction; Harvard Classics; Category...
711,730,Text,1996-11-01,Oliver Twist,en,"Dickens, Charles, 1812-1870",Orphans -- Fiction; London (England) -- Fictio...,PR,Category: Novels; Category: Classics of Litera...
767,786,Text,1997-01-01,Hard Times,en,"Dickens, Charles, 1812-1870",England -- Fiction; Domestic fiction; Social p...,PR,Category: Novels; Category: British Literature
864,883,Text,2006-04-27,Our Mutual Friend,en,"Dickens, Charles, 1812-1870",Satire; London (England) -- Fiction; Inheritan...,PR,Category: Novels; Category: British Literature
1004,1023,Text,1997-08-01,Bleak House,en,"Dickens, Charles, 1812-1870",London (England) -- Fiction; Inheritance and s...,PR,Category: Novels; Category: Classics of Litera...
1378,1400,Text,1998-07-01,Great Expectations,en,"Dickens, Charles, 1812-1870",Orphans -- Fiction; England -- Fiction; Reveng...,PR,Best Books Ever Listings; Category: Novels; Ca...
6011,6053,Text,2004-07-01,"Evelina, Or, the History of a Young Lady's Ent...",en,"Burney, Fanny, 1752-1840",Satire; Humorous stories; London (England) -- ...,PR,Category: Novels; Category: British Literature
6551,6593,Text,2004-09-01,"History of Tom Jones, a Foundling",en,"Fielding, Henry, 1707-1754",Humorous stories; England -- Fiction; Bildungs...,PR,Best Books Ever Listings; Harvard Classics; Mo...
9111,9182,Text,2005-10-01,Villette,en,"Brontë, Charlotte, 1816-1855",Autobiographical fiction; Love stories; Women ...,PR,Category: Novels; Category: British Literature
9225,9296,Text,2005-11-01,Clarissa Harlowe; or the history of a young la...,en,"Richardson, Samuel, 1689-1761",England -- Fiction; Psychological fiction; Epi...,PR,Category: Novels; Category: British Literature


In [56]:
# -----------------------------
# 3. Extract all unique subjects
# -----------------------------

# Split semicolon-separated subject strings
subjects_long = (
    pg_en[["gutenberg_id", "title", "author", "subjects"]]
    .assign(subject=pg_en["subjects"].str.split(";"))
    .explode("subject")
)

# Clean subject strings
subjects_long["subject"] = (
    subjects_long["subject"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove empty subjects
subjects_long = subjects_long[subjects_long["subject"] != ""].copy()

# Count frequency of each unique subject
unique_subjects = (
    subjects_long
    .groupby("subject", as_index=False)
    .agg(
        n_books=("gutenberg_id", "nunique"),
        example_titles=("title", lambda x: "; ".join(list(dict.fromkeys(x.dropna().astype(str)))[:5]))
    )
    .sort_values("n_books", ascending=False)
    .reset_index(drop=True)
)

print(unique_subjects.shape)
unique_subjects.head(5)

(36787, 3)


,subject,n_books,example_titles
0,Science fiction,3111,The Time Machine; The war of the worlds; The s...
1,Short stories,2972,Violists; Flower Fables; The Jungle Book; Stor...
2,Fiction,1399,Stories from the Old Attic; The Early Short Fi...
3,Adventure stories,1387,"Moby-Dick; or, The Whale; The Scarlet Pimperne..."
4,Conduct of life -- Juvenile fiction,1025,The Marvelous Land of Oz; Cast Upon the Breake...


In [57]:
# -----------------------------
# 5. Optional: inspect unique bookshelves
# -----------------------------

bookshelves_long = (
    pg_en[["gutenberg_id", "title", "author", "bookshelves"]]
    .assign(bookshelf=pg_en["bookshelves"].str.split(";"))
    .explode("bookshelf")
)

bookshelves_long["bookshelf"] = (
    bookshelves_long["bookshelf"]
    .fillna("")
    .astype(str)
    .str.strip()
)

bookshelves_long = bookshelves_long[bookshelves_long["bookshelf"] != ""].copy()

unique_bookshelves = (
    bookshelves_long
    .groupby("bookshelf", as_index=False)
    .agg(
        n_books=("gutenberg_id", "nunique"),
        example_titles=("title", lambda x: "; ".join(list(dict.fromkeys(x.dropna().astype(str)))[:5]))
    )
    .sort_values("n_books", ascending=False)
    .reset_index(drop=True)
)

unique_bookshelves.to_csv("gutenberg_unique_bookshelves.csv", index=False)

unique_bookshelves.head(5)

,bookshelf,n_books,example_titles
0,Category: Novels,18505,Alice's Adventures in Wonderland; Through the ...
1,Category: British Literature,9425,Alice's Adventures in Wonderland; Through the ...
2,Category: American Literature,7595,"Moby-Dick; or, The Whale; The Song of Hiawatha..."
3,Category: Adventure,7370,"Moby-Dick; or, The Whale; Peter Pan; The Wonde..."
4,Category: Children & Young Adult Reading,6356,Alice's Adventures in Wonderland; Through the ...


## Exclude non fiction, translations, and short stories

In [58]:
df = pg_en.copy()

# Make sure these columns exist and are strings
for col in ["subjects", "bookshelves", "title", "author"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str)

# Combine subjects + bookshelves for filtering
df["tag_text"] = (
    df["subjects"] + "; " + df["bookshelves"]
).str.lower()

In [59]:
# -----------------------------
# Remove translations into English
# -----------------------------

translation_pattern = (
    r"translations into english|"
    r"translated from|"
    r"-- translations"
)

df["exclude_translation"] = df["tag_text"].str.contains(
    translation_pattern,
    regex=True,
    na=False
)

In [60]:
# -----------------------------
# Remove clearly non-fiction / non-novel forms
# Conservative: only obvious form categories
# -----------------------------

non_novel_form_pattern = (
    r"\bpoetry\b|"
    r"\bpoems\b|"
    r"\bdrama\b|"
    r"\bplays\b|"
    r"\bcomedy plays\b|"
    r"\bconstitution\b|"
    r"\bperiodicals\b|"
    r"\bnewspapers\b|"
    r"\bindexes\b|"
    r"\bbibliography\b|"
    r"\bcatalogs?\b|"
    r"\bdictionaries\b|"
    r"\bencyclopedias\b|"
    r"\bessays\b|"
   # r"\bletters\b|"
    r"\bcorrespondence\b|"
   # r"\bdiaries\b|"
    r"\bjournals\b|"
    r"\bautobiography\b|"
    r"\bbiography\b|"
    r"\bmemoirs\b|"
    r"\bsermons\b|"
    r"\bspeeches\b|"
    r"\blectures\b|"
    r"\btextbooks\b|"
    r"\bhandbooks\b|"
    r"\bmanuals\b|"
    r"\bdescription and travel\b|"
    r"\bpersonal narratives\b|"
   # r"\bconduct of life\b(?!\s*--\s*fiction)|"
    r"\btime management\b|"
    r"\bmental efficiency\b|"
    r"\bjournalism\b|"
    r"\bworld history\b|"
    r"\bhistory -- philosophy\b|"
    r"\bhistory -- study and teaching\b|"
    r"\bsocial sciences\b|"
    #r"\bsocial problems\b|"
    r"\bsocialism\b|"
    r"\bcommunism\b|"
    r"\beducation\b|"
    r"\bgames\b|"
    r"\bwar games\b|"
    r"\breligion\b|"
    r"\bgod \(christianity\)\b|"
    r"\bjournalists?\b|"
    r"\bquestions and answers\b"

)

df["exclude_non_novel_form"] = df["tag_text"].str.contains(
    non_novel_form_pattern,
    regex=True,
    na=False
)

In [61]:
# Short stories
short_story_pattern = (
    r"\bshort stories\b|"
    r"\bshort stories,"
)

df["is_short_story"] = df["tag_text"].str.contains(
    short_story_pattern,
    regex=True,
    na=False
)

In [62]:
# Main exclusion rule
df["exclude_definitely_out"] = (
    df["exclude_translation"] |
    df["exclude_non_novel_form"] |
    df["is_short_story"]
)

pg_fiction_pool_v1 = df[~df["exclude_definitely_out"]].copy()
pg_removed_v1 = df[df["exclude_definitely_out"]].copy()

print("Original English records:", len(df))
print("Removed:", len(pg_removed_v1))
print("Remaining:", len(pg_fiction_pool_v1))

print("\nRemoval reasons:")
print(pd.Series({
    "translation": df["exclude_translation"].sum(),
    "non_novel_form": df["exclude_non_novel_form"].sum(),
    "short stories": df["is_short_story"].sum()
    #"either": df["exclude_definitely_out"].sum(),
}))

Original English records: 61583
Removed: 27540
Remaining: 34043

Removal reasons:
translation        1580
non_novel_form    21895
short stories      5808
dtype: int64


In [63]:
short_story_pattern = r"\bshort stories\b"

df = pg_en.copy()

for col in ["subjects", "bookshelves"]:
    df[col] = df[col].fillna("").astype(str)

df["subject_text"] = df["subjects"].str.lower()
df["bookshelf_text"] = df["bookshelves"].str.lower()
df["tag_text"] = df["subject_text"] + "; " + df["bookshelf_text"]

df["is_short_story_subject"] = df["subject_text"].str.contains(
    short_story_pattern, regex=True, na=False
)

df["is_short_story_bookshelf"] = df["bookshelf_text"].str.contains(
    short_story_pattern, regex=True, na=False
)

df["is_short_story_any"] = df["tag_text"].str.contains(
    short_story_pattern, regex=True, na=False
)

print("Rows matched by subjects only:", df["is_short_story_subject"].sum())
print("Rows matched by bookshelves only:", df["is_short_story_bookshelf"].sum())
print("Rows matched by subjects OR bookshelves:", df["is_short_story_any"].sum())

print("Unique books matched by subjects only:", df.loc[df["is_short_story_subject"], "gutenberg_id"].nunique())
print("Unique books matched by bookshelves only:", df.loc[df["is_short_story_bookshelf"], "gutenberg_id"].nunique())
print("Unique books matched by subjects OR bookshelves:", df.loc[df["is_short_story_any"], "gutenberg_id"].nunique())

Rows matched by subjects only: 3799
Rows matched by bookshelves only: 3618
Rows matched by subjects OR bookshelves: 5808
Unique books matched by subjects only: 3799
Unique books matched by bookshelves only: 3618
Unique books matched by subjects OR bookshelves: 5808


In [64]:
subjects_long_after_filter = (
    pg_fiction_pool_v1[["gutenberg_id", "title", "author", "subjects"]]
    .assign(subject=pg_fiction_pool_v1["subjects"].str.split(";"))
    .explode("subject")
)

subjects_long_after_filter["subject"] = (
    subjects_long_after_filter["subject"]
    .fillna("")
    .astype(str)
    .str.strip()
)

subjects_long_after_filter = subjects_long_after_filter[
    subjects_long_after_filter["subject"] != ""
].copy()

unique_subjects_after_filter = (
    subjects_long_after_filter
    .groupby("subject", as_index=False)
    .agg(
        n_books=("gutenberg_id", "nunique"),
        example_titles=("title", lambda x: "; ".join(list(dict.fromkeys(x.dropna().astype(str)))[:5]))
    )
    .sort_values("n_books", ascending=False)
    .reset_index(drop=True)
)

unique_subjects_after_filter.head(10)

,subject,n_books,example_titles
0,Science fiction,1283,The Time Machine; The war of the worlds; The s...
1,Adventure stories,1264,"Moby-Dick; or, The Whale; The Scarlet Pimperne..."
2,Fiction,1137,The Count's Millions; Margret Howth: A Story o...
3,Conduct of life -- Juvenile fiction,870,The Marvelous Land of Oz; Cast Upon the Breake...
4,Detective and mystery stories,843,The Haunted Hotel: A Mystery of Modern Venice;...
5,Love stories,783,Far from the Madding Crowd; Ivanhoe: A Romance...
6,Man-woman relationships -- Fiction,758,Tess of the d'Urbervilles: A Pure Woman; Summe...
7,Historical fiction,605,O Pioneers!; The Scarlet Letter; The Scarlet P...
8,Western stories,550,My Ántonia; Remember the Alamo; Desert Gold; J...
9,Friendship -- Juvenile fiction,443,The Marvelous Land of Oz; The Golden Road; Cas...


In [65]:
pg_fiction_pool_v1.iloc[1]

,10
gutenberg_id,11
type,Text
gutenberg_issued,2008-06-27
title,Alice's Adventures in Wonderland
language,en
author,"Carroll, Lewis, 1832-1898"
subjects,Fantasy fiction; Children's stories; Imaginary...
locc,PR; PZ
bookshelves,Children's Literature; Category: Children & Yo...
tag_text,fantasy fiction; children's stories; imaginary...


In [138]:
added_canonical_id = [599, 730, 786, 883, 1023, 1400, 6053, 6593, 9182, 9296]
pg_removed_v1[pg_removed_v1['gutenberg_id'].isin(added_canonical_id)]


,gutenberg_id,type,gutenberg_issued,title,language,author,subjects,locc,bookshelves,tag_text,exclude_translation,exclude_non_novel_form,is_short_story,exclude_definitely_out
767,786,Text,1997-01-01,Hard Times,en,"Dickens, Charles, 1812-1870",England -- Fiction; Domestic fiction; Social p...,PR,Category: Novels; Category: British Literature,england -- fiction; domestic fiction; social p...,False,True,False,True


In [150]:
pg_fiction_pool_v1 = pd.concat([pg_fiction_pool_v1, pg_removed_v1[pg_removed_v1['gutenberg_id']==786]])
len(pg_fiction_pool_v1)

34047

In [151]:
pg_fiction_pool_v1[pg_fiction_pool_v1['gutenberg_id']== 786]

,gutenberg_id,type,gutenberg_issued,title,language,author,subjects,locc,bookshelves,tag_text,exclude_translation,exclude_non_novel_form,is_short_story,exclude_definitely_out
767,786,Text,1997-01-01,Hard Times,en,"Dickens, Charles, 1812-1870",England -- Fiction; Domestic fiction; Social p...,PR,Category: Novels; Category: British Literature,england -- fiction; domestic fiction; social p...,False,True,False,True


In [152]:
added_canonical_id = [599, 730, 786, 883, 1023, 1400, 6053, 6593, 9182, 9296]
pg_fiction_pool_v1[pg_fiction_pool_v1['gutenberg_id'].isin(added_canonical_id)]

,gutenberg_id,type,gutenberg_issued,title,language,author,subjects,locc,bookshelves,tag_text,exclude_translation,exclude_non_novel_form,is_short_story,exclude_definitely_out
580,599,Text,1996-07-01,Vanity Fair,en,"Thackeray, William Makepeace, 1811-1863",Satire; England -- Fiction; Married women -- F...,PR,Historical Fiction; Harvard Classics; Category...,satire; england -- fiction; married women -- f...,False,False,False,False
711,730,Text,1996-11-01,Oliver Twist,en,"Dickens, Charles, 1812-1870",Orphans -- Fiction; London (England) -- Fictio...,PR,Category: Novels; Category: Classics of Litera...,orphans -- fiction; london (england) -- fictio...,False,False,False,False
864,883,Text,2006-04-27,Our Mutual Friend,en,"Dickens, Charles, 1812-1870",Satire; London (England) -- Fiction; Inheritan...,PR,Category: Novels; Category: British Literature,satire; london (england) -- fiction; inheritan...,False,False,False,False
1004,1023,Text,1997-08-01,Bleak House,en,"Dickens, Charles, 1812-1870",London (England) -- Fiction; Inheritance and s...,PR,Category: Novels; Category: Classics of Litera...,london (england) -- fiction; inheritance and s...,False,False,False,False
1378,1400,Text,1998-07-01,Great Expectations,en,"Dickens, Charles, 1812-1870",Orphans -- Fiction; England -- Fiction; Reveng...,PR,Best Books Ever Listings; Category: Novels; Ca...,orphans -- fiction; england -- fiction; reveng...,False,False,False,False
6011,6053,Text,2004-07-01,"Evelina, Or, the History of a Young Lady's Ent...",en,"Burney, Fanny, 1752-1840",Satire; Humorous stories; London (England) -- ...,PR,Category: Novels; Category: British Literature,satire; humorous stories; london (england) -- ...,False,False,False,False
6551,6593,Text,2004-09-01,"History of Tom Jones, a Foundling",en,"Fielding, Henry, 1707-1754",Humorous stories; England -- Fiction; Bildungs...,PR,Best Books Ever Listings; Harvard Classics; Mo...,humorous stories; england -- fiction; bildungs...,False,False,False,False
9111,9182,Text,2005-10-01,Villette,en,"Brontë, Charlotte, 1816-1855",Autobiographical fiction; Love stories; Women ...,PR,Category: Novels; Category: British Literature,autobiographical fiction; love stories; women ...,False,False,False,False
9225,9296,Text,2005-11-01,Clarissa Harlowe; or the history of a young la...,en,"Richardson, Samuel, 1689-1761",England -- Fiction; Psychological fiction; Epi...,PR,Category: Novels; Category: British Literature,england -- fiction; psychological fiction; epi...,False,False,False,False
767,786,Text,1997-01-01,Hard Times,en,"Dickens, Charles, 1812-1870",England -- Fiction; Domestic fiction; Social p...,PR,Category: Novels; Category: British Literature,england -- fiction; domestic fiction; social p...,False,True,False,True


## Clean Author names

In [153]:
df = pg_fiction_pool_v1.copy()

# -----------------------------
# 1. Prepare subject text
# -----------------------------

df["subjects"] = df["subjects"].fillna("").astype(str)

df["subject_text"] = df["subjects"].str.lower()

In [154]:
# -----------------------------
# 2. Author cleaning functions
# -----------------------------

def split_authors(author_str):
    """
    Split Gutenberg author field into multiple raw authors.
    Gutenberg often separates multiple authors with semicolons.
    """
    if pd.isna(author_str):
        return []

    author_str = str(author_str).strip()
    if author_str == "":
        return []

    authors = [a.strip() for a in author_str.split(";") if a.strip()]
    return authors



# 只保留真正属于姓名的后缀
PERSON_SUFFIXES = {
    "jr", "jr.",
    "sr", "sr.",
    "ii", "iii", "iv", "v"
}


def remove_life_dates(name):
    """
    Aggressively remove birth/death years and uncertain-date markers.

    Examples:
    'Defoe, Daniel, 1661?-1731' -> 'Defoe, Daniel'
    'Houstoun, Mrs., 1815?'     -> 'Houstoun, Mrs.'
    'Author, Name, 1815?-'      -> 'Author, Name'
    'Author, Name, -1898'       -> 'Author, Name'
    """
    if pd.isna(name):
        return ""

    name = str(name).strip()

    # Remove b. 1815, d. 1898, born 1815, died 1898
    name = re.sub(
        r",?\s*\b(?:b|d|born|died)\.?\s*\d{3,4}\??\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    # Remove date ranges and uncertain ranges
    name = re.sub(
        r",?\s*"
        r"(?:"
        r"\d{3,4}\??\s*-\s*\d{0,4}\??"   # 1661?-1731, 1815?-
        r"|-\s*\d{3,4}\??"                # -1898
        r")"
        r"\s*$",
        "",
        name
    )

    # Remove a trailing standalone year, including 1815?
    name = re.sub(
        r",?\s*\d{3,4}\??\s*$",
        "",
        name
    )

    # Aggressive cleanup for any remaining 3- or 4-digit year token
    name = re.sub(
        r"\b\d{3,4}\??\b",
        "",
        name
    )

    # Remove dangling question marks
    name = re.sub(r"\?+", "", name)

    # Clean punctuation and whitespace left behind
    name = re.sub(r"\s+,", ",", name)
    name = re.sub(r",\s*,+", ",", name)
    name = re.sub(r",\s*$", "", name)
    name = re.sub(r"\s+", " ", name)

    return name.strip()


def surname_already_present(given_name, surname):
    """
    Check whether the given-name field already ends with the surname.

    'Edward Bulwer Lytton' + 'Lytton' -> True
    'Emmuska Orczy' + 'Orczy'         -> True
    'Daniel' + 'Defoe'                -> False
    """
    def tokens(text):
        text = str(text).lower()
        text = re.sub(r"[^a-z\s]", " ", text)
        return text.split()

    given_tokens = tokens(given_name)
    surname_tokens = tokens(surname)

    if not surname_tokens or len(given_tokens) < len(surname_tokens):
        return False

    return given_tokens[-len(surname_tokens):] == surname_tokens


def parse_author_name(raw_author, use_parenthetical=False):
    """
    Parse a single Gutenberg author entry.

    use_parenthetical=False:
        'Wells, H. G. (Herbert George), 1866-1946'
        -> 'H. G. Wells'

    use_parenthetical=True:
        -> 'Herbert George Wells'
    """
    if pd.isna(raw_author):
        return ""

    name = remove_life_dates(raw_author)

    if not name:
        return ""

    # Extract expanded name from parentheses
    paren_match = re.search(r"\(([^()]*)\)", name)
    paren_name = paren_match.group(1).strip() if paren_match else ""

    # Remove parentheses before splitting surname/given name
    name_without_paren = re.sub(r"\s*\([^()]*\)", "", name).strip()

    parts = [
        part.strip()
        for part in name_without_paren.split(",")
        if part.strip()
    ]

    if len(parts) < 2:
        return re.sub(r"\s+", " ", name_without_paren).strip()

    surname = parts[0]
    given_name = paren_name if use_parenthetical and paren_name else parts[1]

    # Fields after the first two are usually titles or suffixes.
    # Keep Jr., Sr., II, III, etc.; discard Baron, Baroness, and similar titles.
    suffixes = [
        part for part in parts[2:]
        if part.lower() in PERSON_SUFFIXES
    ]

    if surname_already_present(given_name, surname):
        full_name = given_name
    else:
        full_name = f"{given_name} {surname}"

    if suffixes:
        full_name += " " + " ".join(suffixes)

    return re.sub(r"\s+", " ", full_name).strip()


def clean_author_display(raw_author):
    """Readable author name, retaining initials."""
    return parse_author_name(raw_author, use_parenthetical=False)


def clean_author_query(raw_author):
    """Query-friendly name, using expanded parenthetical name when available."""
    return parse_author_name(raw_author, use_parenthetical=True)


def normalize_author_key(name):
    """
    Lowercase matching key.
    Useful for grouping / de-duplication.
    """
    if pd.isna(name):
        return ""

    name = str(name).lower()
    name = re.sub(r"[^a-z\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

In [155]:
# -----------------------------
# 3. Apply author cleaning
# -----------------------------

df["author_raw"] = df["author"].fillna("").astype(str)

# Keep list of raw authors
df["author_raw_list"] = df["author_raw"].apply(split_authors)

# Primary author = first listed author
df["primary_author_raw"] = df["author_raw_list"].apply(
    lambda x: x[0] if len(x) > 0 else ""
)

df["author_display"] = (
    df["primary_author_raw"]
    .apply(clean_author_display)
)

df["author_query"] = (
    df["primary_author_raw"]
    .apply(clean_author_query)
)



df[
    [
        "gutenberg_id",
        "title",
        "author_raw",
        "primary_author_raw",
        "author_display",
        "author_query"
    ]
].head(10)

,gutenberg_id,title,author_raw,primary_author_raw,author_display,author_query
6,7,The Mayflower Compact,,,,
10,11,Alice's Adventures in Wonderland,"Carroll, Lewis, 1832-1898","Carroll, Lewis, 1832-1898",Lewis Carroll,Lewis Carroll
11,12,Through the Looking-Glass,"Carroll, Lewis, 1832-1898","Carroll, Lewis, 1832-1898",Lewis Carroll,Lewis Carroll
14,15,"Moby-Dick; or, The Whale","Melville, Herman, 1819-1891","Melville, Herman, 1819-1891",Herman Melville,Herman Melville
15,16,Peter Pan,"Barrie, J. M. (James Matthew), 1860-1937","Barrie, J. M. (James Matthew), 1860-1937",J. M. Barrie,James Matthew Barrie
23,24,O Pioneers!,"Cather, Willa, 1873-1947","Cather, Willa, 1873-1947",Willa Cather,Willa Cather
26,27,Far from the Madding Crowd,"Hardy, Thomas, 1840-1928","Hardy, Thomas, 1840-1928",Thomas Hardy,Thomas Hardy
31,32,Herland,"Gilman, Charlotte Perkins, 1860-1935","Gilman, Charlotte Perkins, 1860-1935",Charlotte Perkins Gilman,Charlotte Perkins Gilman
32,33,The Scarlet Letter,"Hawthorne, Nathaniel, 1804-1864","Hawthorne, Nathaniel, 1804-1864",Nathaniel Hawthorne,Nathaniel Hawthorne
33,34,Zen and the Art of the Internet,"Kehoe, Brendan P., 1970-2011","Kehoe, Brendan P., 1970-2011",Brendan P. Kehoe,Brendan P. Kehoe


In [156]:
test_names = pd.Series([
    "Defoe, Daniel, 1661?-1731",
    "Lytton, Edward Bulwer Lytton, Baron, 1803-1873",
    "Orczy, Emmuska Orczy, Baroness, 1865-1947",
    "Wells, H. G. (Herbert George), 1866-1946",
    "Barrie, J. M. (James Matthew), 1860-1937",
    "Smith, Joseph, Jr., 1805-1844",
    "Walk, Charles Edmonds, 1875-1943?; McFall, J. V. [Illustrator]"
])

test_result = pd.DataFrame({
    "author_raw": test_names,
    "author_display": test_names.apply(clean_author_display),
    "author_query": test_names.apply(clean_author_query),
})

test_result

,author_raw,author_display,author_query
0,"Defoe, Daniel, 1661?-1731",Daniel Defoe,Daniel Defoe
1,"Lytton, Edward Bulwer Lytton, Baron, 1803-1873",Edward Bulwer Lytton,Edward Bulwer Lytton
2,"Orczy, Emmuska Orczy, Baroness, 1865-1947",Emmuska Orczy,Emmuska Orczy
3,"Wells, H. G. (Herbert George), 1866-1946",H. G. Wells,Herbert George Wells
4,"Barrie, J. M. (James Matthew), 1860-1937",J. M. Barrie,James Matthew Barrie
5,"Smith, Joseph, Jr., 1805-1844",Joseph Smith Jr.,Joseph Smith Jr.
6,"Walk, Charles Edmonds, 1875-1943?; McFall, J. ...",Charles Edmonds Walk,Charles Edmonds Walk


In [157]:
df=df.loc[:,['gutenberg_id','title','subjects','subject_text','author_raw','author_display','author_query']]
df

,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query
6,7,The Mayflower Compact,"Massachusetts -- History -- New Plymouth, 1620...","massachusetts -- history -- new plymouth, 1620...",,,
10,11,Alice's Adventures in Wonderland,Fantasy fiction; Children's stories; Imaginary...,fantasy fiction; children's stories; imaginary...,"Carroll, Lewis, 1832-1898",Lewis Carroll,Lewis Carroll
11,12,Through the Looking-Glass,Fantasy fiction; Children's stories; Imaginary...,fantasy fiction; children's stories; imaginary...,"Carroll, Lewis, 1832-1898",Lewis Carroll,Lewis Carroll
14,15,"Moby-Dick; or, The Whale",Whaling -- Fiction; Sea stories; Psychological...,whaling -- fiction; sea stories; psychological...,"Melville, Herman, 1819-1891",Herman Melville,Herman Melville
15,16,Peter Pan,Fantasy literature; Peter Pan (Fictitious char...,fantasy literature; peter pan (fictitious char...,"Barrie, J. M. (James Matthew), 1860-1937",J. M. Barrie,James Matthew Barrie
...,...,...,...,...,...,...,...
78997,79184,Pariahs of Piperock,,,"Tuttle, W. C. (Wilbur C.), 1883-1969",W. C. Tuttle,Wilbur C. Tuttle
748,767,Agnes Grey,Autobiographical fiction; England -- Fiction; ...,autobiographical fiction; england -- fiction; ...,"Brontë, Anne, 1820-1849",Anne Brontë,Anne Brontë
748,767,Agnes Grey,Autobiographical fiction; England -- Fiction; ...,autobiographical fiction; england -- fiction; ...,"Brontë, Anne, 1820-1849",Anne Brontë,Anne Brontë
748,767,Agnes Grey,Autobiographical fiction; England -- Fiction; ...,autobiographical fiction; england -- fiction; ...,"Brontë, Anne, 1820-1849",Anne Brontë,Anne Brontë


## Identify domestic & other genres

In [158]:
# -----------------------------
# 2. Domestic fiction rules
# -----------------------------

domestic_patterns = [
    r"\bdomestic fiction\b",
    r"\bcourtship -- fiction\b",
    r"\bmarriage -- fiction\b",
    r"\bmarried people -- fiction\b",
    r"\bman-woman relationships -- fiction\b",
    r"\bfamilies -- fiction\b",
    r"\bvirtue -- fiction\b"
    #r"\bfamily -- fiction\b"
    ]

domestic_pattern = "|".join(domestic_patterns)

df["is_domestic"] = df["subject_text"].str.contains(
    domestic_pattern,
    regex=True,
    na=False
).astype(int)

In [159]:
df["is_domestic"].sum()

np.int64(1549)

In [160]:
# Explode subjects into long format
subjects_long = (
    df[["gutenberg_id", "title", "author_display", "subjects"]]
    .assign(subject=df["subjects"].str.split(";"))
    .explode("subject")
)

subjects_long["subject"] = (
    subjects_long["subject"]
    .fillna("")
    .astype(str)
    .str.strip()
)

subjects_long["subject_lower"] = subjects_long["subject"].str.lower()

domestic_subjects_matched = subjects_long[
    subjects_long["subject_lower"].str.contains(
        domestic_pattern,
        regex=True,
        na=False
    )
].copy()

domestic_subject_summary = (
    domestic_subjects_matched
    .groupby("subject", as_index=False)
    .agg(
        n_books=("gutenberg_id", "nunique"),
        example_titles=("title", lambda x: "; ".join(list(dict.fromkeys(x.dropna().astype(str)))[:5]))
    )
    .sort_values("n_books", ascending=False)
    .reset_index(drop=True)
)

domestic_subject_summary.head(100)

,subject,n_books,example_titles
0,Man-woman relationships -- Fiction,758,Tess of the d'Urbervilles: A Pure Woman; Summe...
1,Domestic fiction,328,O Pioneers!; The House of the Seven Gables; A ...
2,Families -- Fiction,231,Seventeen\r\nA Tale of Youth and Summer Time a...
3,Married people -- Fiction,132,Alexander's Bridge; Middlemarch; McTeague: A S...
4,Marriage -- Fiction,122,Miss Billy — Married; The Shuttle; Anne's Hous...
5,Courtship -- Fiction,94,The American; The Flirt; Pride and Prejudice; ...
6,Poor families -- Fiction,17,A Christmas Carol in Prose; Being a Ghost Stor...
7,Motherless families -- Fiction,12,Persuasion; The Hero of Hill House; Janet's Lo...
8,African American families -- Fiction,7,"Clotelle; Or, The Colored Heroine, a tale of t..."
9,Dysfunctional families -- Fiction,6,Persuasion; Dombey and Son; Eli's Children: Th...


In [161]:
# -----------------------------
# 3. Major genre rules
# -----------------------------

genre_keywords = {
    "detective_mystery": [
        r"\bdetective and mystery stories\b",
        r"\bmystery fiction\b",
        r"\bmystery and detective stories\b",
        r"\bdetective fiction\b",
        r"\bdetective stories\b",
    ],
    "science_fiction": [
        r"\bscience fiction\b",
        r"\bscience-fiction\b",
        r"\bhuman-alien encounters -- fiction\b",
        r"\bspace ships -- fiction\b",
        r"\bspace warfare -- fiction\b",
        r"\blife on other planets -- fiction\b",
        r"\btime travel -- fiction\b",
        r"\butopias -- fiction\b",
        r"\bdystopias -- fiction\b",
    ],
    "adventure": [
        r"\badventure stories\b",
        r"\badventure and adventurers -- fiction\b",
        r"\badventure and adventurers -- juvenile fiction\b",
    ],
    "gothic": [
        r"\bgothic fiction\b",
        r"\bghost stories\b",
        r"\bhorror tales\b",
        r"\bhorror stories\b",
        r"\bsupernatural -- fiction\b",
        r"\boccult fiction\b",
    ],
   # "fantasy": [
   #     r"\bfantasy fiction\b",
   #     r"\bfairy tales\b",
   #     r"\bmagic -- fiction\b",
   #     r"\bfantasy literature\b"
   # ],
    "historical": [
        r"\bhistorical fiction\b",
    ],

   # "romance": [
   #     r"\blove stories\b",
   #     r"\bromance fiction\b",
   # ]

}

genre_cols = []

for genre, patterns in genre_keywords.items():
    col = f"is_{genre}"
    genre_cols.append(col)
    pattern = "|".join(patterns)
    df[col] = df["subject_text"].str.contains(
        pattern,
        regex=True,
        na=False
    ).astype(int)

df["has_major_genre"] = (df[genre_cols].sum(axis=1) > 0).astype(int)

df["genre_tags"] = df.apply(
    lambda row: "; ".join([
        col.replace("is_", "")
        for col in genre_cols
        if row[col] == 1
    ]),
    axis=1
)

In [168]:
added_armstrong_id = [599, 730, 786, 883, 1023, 1400, 6053, 6593, 9182, 9296]
df[df['gutenberg_id'].isin(added_armstrong_id)]

,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query,is_domestic,is_detective_mystery,is_science_fiction,is_adventure,is_gothic,is_historical,has_major_genre,genre_tags,broad_group
580,599,Vanity Fair,Satire; England -- Fiction; Married women -- F...,satire; england -- fiction; married women -- f...,"Thackeray, William Makepeace, 1811-1863",William Makepeace Thackeray,William Makepeace Thackeray,0,0,0,0,0,0,0,,other_non_domestic_non_major_genre
711,730,Oliver Twist,Orphans -- Fiction; London (England) -- Fictio...,orphans -- fiction; london (england) -- fictio...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,0,0,0,0,0,0,0,,other_non_domestic_non_major_genre
864,883,Our Mutual Friend,Satire; London (England) -- Fiction; Inheritan...,satire; london (england) -- fiction; inheritan...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
1004,1023,Bleak House,London (England) -- Fiction; Inheritance and s...,london (england) -- fiction; inheritance and s...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
1378,1400,Great Expectations,Orphans -- Fiction; England -- Fiction; Reveng...,orphans -- fiction; england -- fiction; reveng...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
6011,6053,"Evelina, Or, the History of a Young Lady's Ent...",Satire; Humorous stories; London (England) -- ...,satire; humorous stories; london (england) -- ...,"Burney, Fanny, 1752-1840",Fanny Burney,Fanny Burney,0,0,0,0,0,0,0,,other_non_domestic_non_major_genre
6551,6593,"History of Tom Jones, a Foundling",Humorous stories; England -- Fiction; Bildungs...,humorous stories; england -- fiction; bildungs...,"Fielding, Henry, 1707-1754",Henry Fielding,Henry Fielding,0,0,0,0,0,0,0,,other_non_domestic_non_major_genre
9111,9182,Villette,Autobiographical fiction; Love stories; Women ...,autobiographical fiction; love stories; women ...,"Brontë, Charlotte, 1816-1855",Charlotte Brontë,Charlotte Brontë,0,0,0,0,0,0,0,,other_non_domestic_non_major_genre
9225,9296,Clarissa Harlowe; or the history of a young la...,England -- Fiction; Psychological fiction; Epi...,england -- fiction; psychological fiction; epi...,"Richardson, Samuel, 1689-1761",Samuel Richardson,Samuel Richardson,0,0,0,0,0,0,0,,other_non_domestic_non_major_genre
767,786,Hard Times,England -- Fiction; Domestic fiction; Social p...,england -- fiction; domestic fiction; social p...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only


In [171]:
armstrong_domestic_id = [599, 730, 786, 883, 1023, 1400, 6053, 9182, 9296]
df.loc[df['gutenberg_id'].isin(armstrong_domestic_id), 'is_domestic'] = 1
df[df['gutenberg_id'].isin(added_armstrong_id)]

,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query,is_domestic,is_detective_mystery,is_science_fiction,is_adventure,is_gothic,is_historical,has_major_genre,genre_tags,broad_group
580,599,Vanity Fair,Satire; England -- Fiction; Married women -- F...,satire; england -- fiction; married women -- f...,"Thackeray, William Makepeace, 1811-1863",William Makepeace Thackeray,William Makepeace Thackeray,1,0,0,0,0,0,0,,other_non_domestic_non_major_genre
711,730,Oliver Twist,Orphans -- Fiction; London (England) -- Fictio...,orphans -- fiction; london (england) -- fictio...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,other_non_domestic_non_major_genre
864,883,Our Mutual Friend,Satire; London (England) -- Fiction; Inheritan...,satire; london (england) -- fiction; inheritan...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
1004,1023,Bleak House,London (England) -- Fiction; Inheritance and s...,london (england) -- fiction; inheritance and s...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
1378,1400,Great Expectations,Orphans -- Fiction; England -- Fiction; Reveng...,orphans -- fiction; england -- fiction; reveng...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
6011,6053,"Evelina, Or, the History of a Young Lady's Ent...",Satire; Humorous stories; London (England) -- ...,satire; humorous stories; london (england) -- ...,"Burney, Fanny, 1752-1840",Fanny Burney,Fanny Burney,1,0,0,0,0,0,0,,other_non_domestic_non_major_genre
6551,6593,"History of Tom Jones, a Foundling",Humorous stories; England -- Fiction; Bildungs...,humorous stories; england -- fiction; bildungs...,"Fielding, Henry, 1707-1754",Henry Fielding,Henry Fielding,0,0,0,0,0,0,0,,other_non_domestic_non_major_genre
9111,9182,Villette,Autobiographical fiction; Love stories; Women ...,autobiographical fiction; love stories; women ...,"Brontë, Charlotte, 1816-1855",Charlotte Brontë,Charlotte Brontë,1,0,0,0,0,0,0,,other_non_domestic_non_major_genre
9225,9296,Clarissa Harlowe; or the history of a young la...,England -- Fiction; Psychological fiction; Epi...,england -- fiction; psychological fiction; epi...,"Richardson, Samuel, 1689-1761",Samuel Richardson,Samuel Richardson,1,0,0,0,0,0,0,,other_non_domestic_non_major_genre
767,786,Hard Times,England -- Fiction; Domestic fiction; Social p...,england -- fiction; domestic fiction; social p...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only


In [178]:
# -----------------------------
# 4. Assign broad group
# -----------------------------

def assign_group(row):
    domestic = row["is_domestic"] == 1
    genre = row["has_major_genre"] == 1

    if domestic and genre:
        return "domestic_and_genre"
    elif domestic:
        return "domestic_only"
    elif genre:
        return "major_genre_non_domestic"
    else:
        return "other_non_domestic_non_major_genre"

df["broad_group"] = df.apply(assign_group, axis=1)

df["broad_group"].value_counts()

,count
broad_group,
other_non_domestic_non_major_genre,27947
major_genre_non_domestic,4546
domestic_only,1354
domestic_and_genre,200


In [179]:
genre_counts = df[genre_cols].sum().sort_values(ascending=False)
genre_counts

,0
is_adventure,1607
is_detective_mystery,1430
is_science_fiction,1360
is_historical,616
is_gothic,162


In [180]:
df_domestic = df[df["is_domestic"] == 1].copy()

df_major_genre_non_domestic = df[
    (df["is_domestic"] == 0) &
    (df["has_major_genre"] == 1)
].copy()

df_other = df[
    (df["is_domestic"] == 0) &
    (df["has_major_genre"] == 0)
].copy()

print("Domestic:", len(df_domestic))
print("Major genre non-domestic:", len(df_major_genre_non_domestic))
print("Other:", len(df_other))

Domestic: 1554
Major genre non-domestic: 4546
Other: 27947


In [181]:
tom_jones = df[df['gutenberg_id']==6593]

df_candidate = pd.concat(
    [df_domestic, df_major_genre_non_domestic,tom_jones],
    ignore_index=True
)

# Remove duplicate Gutenberg records
#df_candidate = df_candidate.drop_duplicates(subset=["gutenberg_id"]).copy()

print("Candidate dataset size:", len(df_candidate))

Candidate dataset size: 6101


In [81]:
df_candidate.to_csv("pg_candidate_v1.csv", index=False)

In [183]:
added_armstrong_id = [599, 730, 786, 883, 1023, 1400, 6053, 6593, 9182, 9296]
added_armstrong = df_candidate[df_candidate['gutenberg_id'].isin(added_armstrong_id)]
added_armstrong

,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query,is_domestic,is_detective_mystery,is_science_fiction,is_adventure,is_gothic,is_historical,has_major_genre,genre_tags,broad_group
50,599,Vanity Fair,Satire; England -- Fiction; Married women -- F...,satire; england -- fiction; married women -- f...,"Thackeray, William Makepeace, 1811-1863",William Makepeace Thackeray,William Makepeace Thackeray,1,0,0,0,0,0,0,,domestic_only
53,730,Oliver Twist,Orphans -- Fiction; London (England) -- Fictio...,orphans -- fiction; london (england) -- fictio...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
57,883,Our Mutual Friend,Satire; London (England) -- Fiction; Inheritan...,satire; london (england) -- fiction; inheritan...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
63,1023,Bleak House,London (England) -- Fiction; Inheritance and s...,london (england) -- fiction; inheritance and s...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
75,1400,Great Expectations,Orphans -- Fiction; England -- Fiction; Reveng...,orphans -- fiction; england -- fiction; reveng...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
245,6053,"Evelina, Or, the History of a Young Lady's Ent...",Satire; Humorous stories; London (England) -- ...,satire; humorous stories; london (england) -- ...,"Burney, Fanny, 1752-1840",Fanny Burney,Fanny Burney,1,0,0,0,0,0,0,,domestic_only
340,9182,Villette,Autobiographical fiction; Love stories; Women ...,autobiographical fiction; love stories; women ...,"Brontë, Charlotte, 1816-1855",Charlotte Brontë,Charlotte Brontë,1,0,0,0,0,0,0,,domestic_only
341,9296,Clarissa Harlowe; or the history of a young la...,England -- Fiction; Psychological fiction; Epi...,england -- fiction; psychological fiction; epi...,"Richardson, Samuel, 1689-1761",Samuel Richardson,Samuel Richardson,1,0,0,0,0,0,0,,domestic_only
1553,786,Hard Times,England -- Fiction; Domestic fiction; Social p...,england -- fiction; domestic fiction; social p...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only
6100,6593,"History of Tom Jones, a Foundling",Humorous stories; England -- Fiction; Bildungs...,humorous stories; england -- fiction; bildungs...,"Fielding, Henry, 1707-1754",Henry Fielding,Henry Fielding,0,0,0,0,0,0,0,,other_non_domestic_non_major_genre


In [184]:
added_armstrong.to_csv("added_armstrong.csv", index=False)

## Explore other genres

In [82]:
# -----------------------------
# 5. Inspect subjects in remaining other group
# -----------------------------

other_subjects_long = (
    df_other[["gutenberg_id", "title", "author_display", "subjects"]]
    .assign(subject=df_other["subjects"].str.split(";"))
    .explode("subject")
)

other_subjects_long["subject"] = (
    other_subjects_long["subject"]
    .fillna("")
    .astype(str)
    .str.strip()
)

other_subjects_long = other_subjects_long[
    other_subjects_long["subject"] != ""
].copy()

other_unique_subjects = (
    other_subjects_long
    .groupby("subject", as_index=False)
    .agg(
        n_books=("gutenberg_id", "nunique"),
        example_titles=("title", lambda x: "; ".join(list(dict.fromkeys(x.dropna().astype(str)))[:5]))
    )
    .sort_values("n_books", ascending=False)
    .reset_index(drop=True)
)

other_unique_subjects.head(30)

,subject,n_books,example_titles
0,Fiction,1008,Margret Howth: A Story of To-day; Danny's Own ...
1,Conduct of life -- Juvenile fiction,662,The Marvelous Land of Oz; Cast Upon the Breake...
2,Love stories,498,Far from the Madding Crowd; The Return of the ...
3,Western stories,456,Remember the Alamo; Desert Gold; Jean of the L...
4,Christian life -- Juvenile fiction,372,The Pedler of Dust Sticks; Richard Dare's Vent...
5,Friendship -- Juvenile fiction,324,The Marvelous Land of Oz; The Golden Road; Cas...
6,English fiction -- 19th century,312,The Gathering of Brother Hilarius; The Perils ...
7,Siblings -- Juvenile fiction,288,The Bobbsey Twins in the Country; The Bobbsey ...
8,Children -- Conduct of life -- Juvenile fiction,288,Little Lord Fauntleroy; Five Little Peppers an...
9,Young women -- Fiction,242,The Song of the Lark; The Voyage Out; Daddy-Lo...


In [83]:
df_check = df.copy()

df_check["subjects"] = (
    df_check["subjects"]
    .fillna("")
    .astype(str)
)

# One row per book-subject pair
subjects_long = (
    df_check[
        [
            "gutenberg_id",
            "title",
            "author_display",
            "subjects",
        ]
    ]
    .assign(subject=lambda x: x["subjects"].str.split(";"))
    .explode("subject")
)

subjects_long["subject"] = (
    subjects_long["subject"]
    .fillna("")
    .astype(str)
    .str.strip()
)

subjects_long = subjects_long[
    subjects_long["subject"] != ""
].copy()

In [84]:
def inspect_subject(subject_name, df=df_check, subjects_long=subjects_long):
    ids = subjects_long.loc[
        subjects_long["subject"].eq(subject_name),
        "gutenberg_id"
    ].unique()

    subset = df[
        df["gutenberg_id"].isin(ids)
    ].copy()

    print(f"Subject: {subject_name}")
    print(f"Books: {subset['gutenberg_id'].nunique()}")
    print(f"Authors: {subset['author_display'].nunique()}")

    if "is_domestic" in subset.columns:
        print("Domestic:", subset["is_domestic"].sum())

    if "has_major_genre" in subset.columns:
        print("Major genre:", subset["has_major_genre"].sum())

    print("\nTop authors:")
    display(
        subset.groupby(
            "author_display",
            dropna=False
        )["gutenberg_id"]
        .nunique()
        .sort_values(ascending=False)
        .rename("n_books")
        .reset_index()
        .head(30)
    )

    print("\nWorks:")
    display(
        subset[
            [
                "gutenberg_id",
                "title",
                "author_display",
                "subjects",
                "is_domestic",
                "genre_tags",
                "has_major_genre",
            ]
        ]
        .sort_values(
            ["author_display", "title"]
        )
        .reset_index(drop=True)
    )

    return subset

In [85]:
df_all_fiction = df.copy()

df_all_fiction["subjects"] = (
    df_all_fiction["subjects"]
    .fillna("")
    .astype(str)
)

# Split subjects exactly
subject_lists = df_all_fiction["subjects"].apply(
    lambda x: [s.strip() for s in x.split(";") if s.strip()]
)

df_all_fiction["is_english_fiction_19c"] = subject_lists.apply(
    lambda xs: int("English fiction -- 19th century" in xs)
)



df_all_fiction["is_general_19c_english_candidate"] = (
    (df_all_fiction["is_english_fiction_19c"] == 1)
    & (df_all_fiction["is_domestic"] == 0)
    & (df_all_fiction["has_major_genre"] == 0)
   # & (df_all_fiction["is_romance_related"] == 0)
).astype(int)

In [86]:
df_general_19c = df_all_fiction[
    df_all_fiction["is_general_19c_english_candidate"] == 1
].copy()

print("General 19c English candidates:", len(df_general_19c))
print("Authors:", df_general_19c["author_display"].nunique())

General 19c English candidates: 312
Authors: 57


In [87]:
author_counts_19c = (
    df_general_19c
    .groupby("author_display")["gutenberg_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_books")
    .reset_index()
)

author_counts_19c.head(30)

,author_display,n_books
0,Edward Bulwer Lytton,75
1,George Meredith,53
2,M. E. Braddon,20
3,George Gissing,15
4,Florence Marryat,14
5,William Clark Russell,13
6,Henry Wood,8
7,B. L. Farjeon,8
8,S. Baring-Gould,8
9,T. W. Speight,8


In [88]:
df_general_19c["broad_group"] = "general_19c_english_candidate"

df_candidate = pd.concat(
    [
        df_candidate,
        df_general_19c,
    ],
    ignore_index=True
).drop_duplicates(
    subset=["gutenberg_id"]
)

df_candidate["broad_group"].value_counts()

,count
broad_group,
major_genre_non_domestic,4546
domestic_only,1348
general_19c_english_candidate,312
domestic_and_genre,200


In [89]:
len(df_candidate)

6406

In [90]:
df_young_women = inspect_subject(
    "Young women -- Fiction"
)

Subject: Young women -- Fiction
Books: 409
Authors: 269
Domestic: 130
Major genre: 45

Top authors:


,author_display,n_books
0,Jane Austen,11
1,Henry James,8
2,Alex. McVeigh Miller,7
3,George de Horne Vaizey,6
4,C. N. Williamson,6
5,Mrs. Oliphant,5
6,Humphry Ward,5
7,William Dean Howells,5
8,M. E. Braddon,5
9,Elisabeth Sanxay Holding,4



Works:


,gutenberg_id,title,author_display,subjects,is_domestic,genre_tags,has_major_genre
0,39730,The Girl Who Had Nothing,A. M. Williamson,Young women -- Fiction; Man-woman relationship...,1,,0
1,38083,A Mere Chance: A Novel. Vol. 1,Ada Cambridge,Young women -- Fiction; Australia -- Social li...,0,,0
2,38084,A Mere Chance: A Novel. Vol. 2,Ada Cambridge,Young women -- Fiction; Australia -- Social li...,0,,0
3,38085,A Mere Chance: A Novel. Vol. 3,Ada Cambridge,Young women -- Fiction; Australia -- Social li...,0,,0
4,50476,The Three Miss Kings: An Australian Story,Ada Cambridge,Young women -- Fiction; Sisters -- Fiction; Au...,0,,0
...,...,...,...,...,...,...,...
404,3405,Ragged Lady — Volume 1,William Dean Howells,Young women -- Fiction; New England -- Fiction...,0,,0
405,3406,Ragged Lady — Volume 2,William Dean Howells,Young women -- Fiction; New England -- Fiction...,0,,0
406,7797,The Lady of the Aroostook,William Dean Howells,Venice (Italy) -- Fiction; Young women -- Fict...,0,,0
407,9924,Viviette,William John Locke,Young women -- Fiction; Triangles (Interperson...,0,,0


## Add canonical authors

In [91]:
canonical_authors = pd.read_csv("canonical_authors.csv")
canonical_authors.head()

,author,period
0,Charlotte Smith,Romantic
1,Mary Wollstonecraft,Romantic
2,Maria Edgeworth,Romantic
3,Horace Walpole,Romantic
4,Ann Ward Radcliffe,Romantic


In [92]:
df_all_fiction = df.copy()

In [93]:
import unicodedata

def normalize_name(name):
    if pd.isna(name):
        return ""

    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(
        ch for ch in name
        if not unicodedata.combining(ch)
    )

    name = name.lower()
    name = re.sub(r"[^a-z\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name

canonical = canonical_authors.copy()

canonical["canonical_author_key"] = (
    canonical["author"]
    .apply(normalize_name)
)

canonical.head()


,author,period,canonical_author_key
0,Charlotte Smith,Romantic,charlotte smith
1,Mary Wollstonecraft,Romantic,mary wollstonecraft
2,Maria Edgeworth,Romantic,maria edgeworth
3,Horace Walpole,Romantic,horace walpole
4,Ann Ward Radcliffe,Romantic,ann ward radcliffe


In [94]:
df_all_fiction = df_all_fiction.copy()

# ------------------------------------------------
# 1. Canonical author matching
# ------------------------------------------------

df_all_fiction["author_key_normalized"] = (
    df_all_fiction["author_query"]
    .fillna("")
    .apply(normalize_name)
)

df_all_fiction["author_display_normalized"] = (
    df_all_fiction["author_display"]
    .fillna("")
    .apply(normalize_name)
)

canonical_key_set = set(canonical["canonical_author_key"])

df_all_fiction["is_canonical_author"] = (
    df_all_fiction["author_key_normalized"].isin(canonical_key_set)
    | df_all_fiction["author_display_normalized"].isin(canonical_key_set)
).astype(int)

canonical_period_map = (
    canonical
    .drop_duplicates("canonical_author_key")
    .set_index("canonical_author_key")["period"]
    .to_dict()
)

df_all_fiction["canonical_period"] = (
    df_all_fiction["author_key_normalized"]
    .map(canonical_period_map)
    .fillna(
        df_all_fiction["author_display_normalized"]
        .map(canonical_period_map)
    )
)

In [95]:
df_all_fiction.loc[
    df_all_fiction["is_canonical_author"] == 1,
    [
        "author_raw",
        "author_display",
        "author_query",
        "canonical_period",
    ]
].drop_duplicates().head(5)

,author_raw,author_display,author_query,canonical_period
26,"Hardy, Thomas, 1840-1928",Thomas Hardy,Thomas Hardy,Victorian
34,"Wells, H. G. (Herbert George), 1866-1946",H. G. Wells,Herbert George Wells,Edwardian
41,"Stevenson, Robert Louis, 1850-1894",Robert Louis Stevenson,Robert Louis Stevenson,Victorian
45,"Dickens, Charles, 1812-1870; Leech, John, 1817...",Charles Dickens,Charles Dickens,Victorian
81,"Scott, Walter, 1771-1832",Walter Scott,Walter Scott,Romantic


In [96]:
matched_keys = set(
    df_all_fiction.loc[
        df_all_fiction["is_canonical_author"] == 1,
        "author_key_normalized"
    ]
).union(
    set(
        df_all_fiction.loc[
            df_all_fiction["is_canonical_author"] == 1,
            "author_display_normalized"
        ]
    )
)

unmatched_canonical = canonical[
    ~canonical["canonical_author_key"].isin(matched_keys)
][["author", "period"]]

unmatched_canonical

,author,period


In [97]:
# ------------------------------------------------
# 2. Domestic and major genre flags
# ------------------------------------------------

major_genre_cols = [
    "is_detective_mystery",
    "is_science_fiction",
    "is_adventure",
    "is_gothic",
    "is_historical",
]

df_all_fiction[major_genre_cols] = (
    df_all_fiction[major_genre_cols]
    .fillna(0)
    .astype(int)
)

df_all_fiction["has_major_genre"] = (
    df_all_fiction[major_genre_cols].max(axis=1)
).astype(int)

df_all_fiction["is_domestic"] = (
    df_all_fiction["is_domestic"]
    .fillna(0)
    .astype(int)
)

In [98]:
# ------------------------------------------------
# 3. General nineteenth-century English fiction
# ------------------------------------------------

df_all_fiction["subjects"] = (
    df_all_fiction["subjects"]
    .fillna("")
    .astype(str)
)

subject_lists = df_all_fiction["subjects"].apply(
    lambda x: [s.strip() for s in x.split(";") if s.strip()]
)

df_all_fiction["is_english_fiction_19c"] = subject_lists.apply(
    lambda subjects: int(
        "English fiction -- 19th century" in subjects
    )
)

# Raw eligibility for the general 19c group.
# Canonical overlap will be resolved later by group priority.
df_all_fiction["is_general_19c_candidate"] = (
    (df_all_fiction["is_english_fiction_19c"] == 1)
    & (df_all_fiction["is_domestic"] == 0)
    & (df_all_fiction["has_major_genre"] == 0)
).astype(int)

In [99]:
# ------------------------------------------------
# 4. Build the complete candidate dataset
# ------------------------------------------------

keep_candidate = (
    (df_all_fiction["is_domestic"] == 1)
    | (df_all_fiction["has_major_genre"] == 1)
    | (df_all_fiction["is_canonical_author"] == 1)
    | (df_all_fiction["is_general_19c_candidate"] == 1)
)

df_expanded_candidate = df_all_fiction.loc[
    keep_candidate
].copy()

df_expanded_candidate = (
    df_expanded_candidate
    .drop_duplicates(subset=["gutenberg_id"])
    .reset_index(drop=True)
)

print("Expanded candidate size:", len(df_expanded_candidate))

Expanded candidate size: 6700


In [100]:
# ------------------------------------------------
# 5. Assign mutually exclusive final groups
# ------------------------------------------------

domestic = df_expanded_candidate["is_domestic"].eq(1)
major_genre = df_expanded_candidate["has_major_genre"].eq(1)
canonical_flag = df_expanded_candidate["is_canonical_author"].eq(1)
general_19c = df_expanded_candidate["is_general_19c_candidate"].eq(1)

conditions = [
    # Domestic classifications remain primary
    domestic & major_genre,
    domestic & ~major_genre,

    # Genre remains primary even when the author is canonical
    ~domestic & major_genre,

    # Canonical overrides general 19c when both apply
    ~domestic & ~major_genre & canonical_flag,

    # General 19c only when there is no canonical match
    ~domestic & ~major_genre & ~canonical_flag & general_19c,
]

choices = [
    "domestic_adjacent_hybrid",
    "core_domestic",
    "non_domestic_genre",
    "canonical_other",
    "general_19c_english",
]

df_expanded_candidate["auto_group"] = np.select(
    conditions,
    choices,
    default="unclassified",
)

df_expanded_candidate["auto_group"].value_counts(dropna=False)

,count
auto_group,
non_domestic_genre,4546
core_domestic,1348
general_19c_english,309
canonical_other,297
domestic_adjacent_hybrid,200


In [101]:
overlap_check = df_expanded_candidate[
    (df_expanded_candidate["is_canonical_author"] == 1)
    & (df_expanded_candidate["is_general_19c_candidate"] == 1)
][
    [
        "gutenberg_id",
        "title",
        "author_display",
        "is_domestic",
        "has_major_genre",
        "canonical_period",
        "auto_group",
    ]
].sort_values(["author_display", "title"])

overlap_check

,gutenberg_id,title,author_display,is_domestic,has_major_genre,canonical_period,auto_group
284,1406,The Perils of Certain English Prisoners,Charles Dickens,0,0,Victorian,canonical_other
3765,43442,The Golden Butterfly,Walter Besant,0,0,Victorian,canonical_other
626,4062,Gaston de Latour; an unfinished romance,Walter Pater,0,0,Victorian,canonical_other


In [102]:
df_expanded_candidate = df_expanded_candidate.copy()

df_expanded_candidate["subjects"] = (
    df_expanded_candidate["subjects"]
    .fillna("")
    .astype(str)
)

df_expanded_candidate["subject_text"] = (
    df_expanded_candidate["subjects"]
    .str.lower()
)

fiction_pattern = (
    r"\bfiction\b|"
    r"\bnovels?\b|"
    r"\bstories\b|"
    r"\btales\b"
)

df_expanded_candidate["has_fiction_signal"] = (
    df_expanded_candidate["subject_text"]
    .str.contains(
        fiction_pattern,
        regex=True,
        na=False
    )
)

In [103]:
is_canonical_other = (
    df_expanded_candidate["auto_group"] == "canonical_other"
)

keep_row = (
    ~is_canonical_other
    | df_expanded_candidate["has_fiction_signal"]
)

df_expanded_candidate_clean = (
    df_expanded_candidate.loc[keep_row]
    .copy()
    .reset_index(drop=True)
)

print("Before filtering:", len(df_expanded_candidate))
print("After filtering:", len(df_expanded_candidate_clean))
print("Canonical non-fiction removed:", (~keep_row).sum())

Before filtering: 6700
After filtering: 6624
Canonical non-fiction removed: 76


In [104]:
df_removed_canonical_nonfiction = (
    df_expanded_candidate.loc[~keep_row, [
        "gutenberg_id",
        "title",
        "author_display",
        "canonical_period",
        "subjects",
        "auto_group",
    ]]
    .sort_values(["author_display", "title"])
    .reset_index(drop=True)
)

df_removed_canonical_nonfiction

,gutenberg_id,title,author_display,canonical_period,subjects,auto_group
0,12811,The Human Machine,Arnold Bennett,Edwardian,Conduct of life,canonical_other
1,13449,The Plain Man and His Wife,Arnold Bennett,Edwardian,Conduct of life,canonical_other
2,15063,Your United States: Impressions of a first visit,Arnold Bennett,Edwardian,United States -- Social life and customs -- 18...,canonical_other
3,65042,"The British Campaign in France and Flanders, 1914",Arthur Conan Doyle,Victorian,"World War, 1914-1918 -- Campaigns -- Western F...",canonical_other
4,65043,"The British Campaign in France and Flanders, 1915",Arthur Conan Doyle,Victorian,"World War, 1914-1918 -- Campaigns -- Western F...",canonical_other
...,...,...,...,...,...,...
71,26133,"Trial of Duncan Terig, alias Clerk, and Alexan...",Walter Scott,Romantic,"Trials (Murder) -- Scotland; Terig, Duncan -- ...",canonical_other
72,3261,"News from Nowhere; Or, An Epoch of Rest\r\nBei...",William Morris,Victorian,Utopias,canonical_other
73,46241,"Some Notes on Early Woodcut Books, with a Chap...",William Morris,Victorian,Illumination of books and manuscripts; Printin...,canonical_other
74,31596,The Art and Craft of Printing,William Morris,Victorian,Kelmscott Press; Printing -- Great Britain -- ...,canonical_other


In [105]:
# 检查保留下来的 canonical_other
df_expanded_candidate_clean.loc[
    df_expanded_candidate_clean["auto_group"] == "canonical_other",
    [
        "gutenberg_id",
        "title",
        "author_display",
        "canonical_period",
        "subjects",
        "has_fiction_signal",
    ]
].sort_values(["author_display", "title"])

,gutenberg_id,title,author_display,canonical_period,subjects,has_fiction_signal
4860,60483,Antic Hay,Aldous Huxley,Modernist,Satire; Intellectuals -- Fiction; City and tow...,True
370,1999,Crome Yellow,Aldous Huxley,Modernist,Satire; Humorous stories; Intellectuals -- Fic...,True
6198,74437,Those barren leaves,Aldous Huxley,Modernist,Satire; Humorous stories; Italy -- Fiction; Ar...,True
6197,74419,Reuben Sachs,Amy Levy,Victorian,London (England) -- Fiction; Jews -- England -...,True
4666,57447,The Romance of a Shop,Amy Levy,Victorian,London (England) -- Fiction; Women -- Employme...,True
...,...,...,...,...,...,...
457,2565,The Story of the Glittering Plain\r\nWhich Has...,William Morris,Victorian,"Fantasy fiction, English",True
1233,8778,The Water of the Wondrous Isles,William Morris,Victorian,Fantasy fiction,True
61,169,The Well at the World's End: A Tale,William Morris,Victorian,Fantasy fiction; Quests (Expeditions) -- Fiction,True
528,3055,The Wood Beyond the World,William Morris,Victorian,Fantasy fiction,True


In [106]:
df_expanded_candidate_clean

,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query,is_domestic,is_detective_mystery,is_science_fiction,...,genre_tags,broad_group,author_key_normalized,author_display_normalized,is_canonical_author,canonical_period,is_english_fiction_19c,is_general_19c_candidate,auto_group,has_fiction_signal
0,15,"Moby-Dick; or, The Whale",Whaling -- Fiction; Sea stories; Psychological...,whaling -- fiction; sea stories; psychological...,"Melville, Herman, 1819-1891",Herman Melville,Herman Melville,0,0,0,...,adventure,major_genre_non_domestic,herman melville,herman melville,0,NaN,0,0,non_domestic_genre,True
1,24,O Pioneers!,Nebraska -- Fiction; Historical fiction; Front...,nebraska -- fiction; historical fiction; front...,"Cather, Willa, 1873-1947",Willa Cather,Willa Cather,1,0,0,...,historical,domestic_and_genre,willa cather,willa cather,0,NaN,0,0,domestic_adjacent_hybrid,True
2,27,Far from the Madding Crowd,Didactic fiction; Love stories; Triangles (Int...,didactic fiction; love stories; triangles (int...,"Hardy, Thomas, 1840-1928",Thomas Hardy,Thomas Hardy,0,0,0,...,,other_non_domestic_non_major_genre,thomas hardy,thomas hardy,1,Victorian,0,0,canonical_other,True
3,32,Herland,Utopias -- Fiction; Women -- Fiction; Utopian ...,utopias -- fiction; women -- fiction; utopian ...,"Gilman, Charlotte Perkins, 1860-1935",Charlotte Perkins Gilman,Charlotte Perkins Gilman,0,0,1,...,science_fiction,major_genre_non_domestic,charlotte perkins gilman,charlotte perkins gilman,0,NaN,0,0,non_domestic_genre,True
4,33,The Scarlet Letter,Adultery -- Fiction; Historical fiction; Reven...,adultery -- fiction; historical fiction; reven...,"Hawthorne, Nathaniel, 1804-1864",Nathaniel Hawthorne,Nathaniel Hawthorne,0,0,0,...,historical,major_genre_non_domestic,nathaniel hawthorne,nathaniel hawthorne,0,NaN,0,0,non_domestic_genre,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6619,79129,Thorstein of the mere,Historical fiction; Northmen -- Fiction; Lake ...,historical fiction; northmen -- fiction; lake ...,"Collingwood, W. G. (William Gershom), 1854-1932",W. G. Collingwood,William Gershom Collingwood,0,0,0,...,historical,major_genre_non_domestic,william gershom collingwood,w g collingwood,0,NaN,0,0,non_domestic_genre,True
6620,79136,Hashknife of the Diamond H,Western stories; Mystery fiction; Ranch life -...,western stories; mystery fiction; ranch life -...,"Tuttle, W. C. (Wilbur C.), 1883-1969",W. C. Tuttle,Wilbur C. Tuttle,0,1,0,...,detective_mystery,major_genre_non_domestic,wilbur c tuttle,w c tuttle,0,NaN,0,0,non_domestic_genre,True
6621,79145,Tangled trails,Western stories; Mystery fiction; Sheriffs -- ...,western stories; mystery fiction; sheriffs -- ...,"Tuttle, W. C. (Wilbur C.), 1883-1969",W. C. Tuttle,Wilbur C. Tuttle,0,1,0,...,detective_mystery,major_genre_non_domestic,wilbur c tuttle,w c tuttle,0,NaN,0,0,non_domestic_genre,True
6622,79160,The man in the dark,Detective and mystery stories; London (England...,detective and mystery stories; london (england...,"Ferguson, J. A. (John Alexander), 1871-1952",J. A. Ferguson,John Alexander Ferguson,0,1,0,...,detective_mystery,major_genre_non_domestic,john alexander ferguson,j a ferguson,0,NaN,0,0,non_domestic_genre,True


In [107]:
df_expanded_candidate_clean["auto_group"].value_counts()

,count
auto_group,
non_domestic_genre,4546
core_domestic,1348
general_19c_english,309
canonical_other,221
domestic_adjacent_hybrid,200


In [108]:
df_expanded_candidate_clean.to_csv("pg_candidate_v3.csv", index=False)

In [112]:
added_canonical_id = [599, 730, 786, 883, 1023, 1400, 6053, 6593, 9182, 9296]
df_candidate[df_candidate['gutenberg_id'].isin(added_canonical_id)]

,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query,is_domestic,is_detective_mystery,is_science_fiction,is_adventure,is_gothic,is_historical,has_major_genre,genre_tags,broad_group,is_english_fiction_19c,is_general_19c_english_candidate
55,883,Our Mutual Friend,Satire; London (England) -- Fiction; Inheritan...,satire; london (england) -- fiction; inheritan...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only,NaN,NaN
61,1023,Bleak House,London (England) -- Fiction; Inheritance and s...,london (england) -- fiction; inheritance and s...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only,NaN,NaN
73,1400,Great Expectations,Orphans -- Fiction; England -- Fiction; Reveng...,orphans -- fiction; england -- fiction; reveng...,"Dickens, Charles, 1812-1870",Charles Dickens,Charles Dickens,1,0,0,0,0,0,0,,domestic_only,NaN,NaN
